In [1]:
import pandas as pd
import timeit
import urllib.request
import zipfile
import os
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
csv_path = "household_power_consumption.txt"

if not os.path.exists(csv_path):
    print("Завантаження архіву...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()

df = pd.read_csv(csv_path, sep=';', na_values=['?'], low_memory=False)
df = df.dropna()
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time

Завантаження архіву...


In [2]:
def filter_1(data):
    return data[data['Global_active_power'] > 5.0]

def filter_2(data):
    res = data[(data['Global_intensity'] >= 19.0) & (data['Global_intensity'] <= 20.0)]
    return res[res['Sub_metering_2'] > res['Sub_metering_3']]

def filter_3(data):
    sample = data.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def filter_4(data):
    res = data[data['Time'] >= pd.to_datetime('18:00:00').time()]
    res = res[res['Global_active_power'] > 6.0]
    res = res[(res['Sub_metering_2'] > res['Sub_metering_1']) & (res['Sub_metering_2'] > res['Sub_metering_3'])]
    
    half1 = res.iloc[:len(res)//2]
    half2 = res.iloc[len(res)//2:]
    
    return pd.concat([half1.iloc[2::3], half2.iloc[3::4]])

time_1 = timeit.timeit(lambda: filter_1(df), number=5)
print(f"Час виконання filter_1 (5 разів): {time_1:.4f} сек")

display(filter_4(df).head())

Час виконання filter_1 (5 разів): 0.0177 сек


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
43,2006-12-16,18:07:00,6.474,0.144,231.85,27.8,0.0,37.0,16.0
3007,2006-12-18,19:31:00,6.158,0.442,229.08,27.0,0.0,36.0,0.0
17497,2006-12-28,21:01:00,7.062,0.270,235.76,30.2,2.0,65.0,17.0
17500,2006-12-28,21:04:00,7.376,0.238,234.67,31.4,1.0,72.0,17.0
17503,2006-12-28,21:07:00,7.248,0.000,235.34,30.8,1.0,72.0,17.0


In [3]:
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[['Global_active_power', 'Global_intensity']] = scaler.fit_transform(df[['Global_active_power', 'Global_intensity']])

pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Коефіцієнт Пірсона: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_corr:.4f}")

df['DayOfWeek'] = df['Date'].dt.day_name()
df_encoded = pd.get_dummies(df, columns=['DayOfWeek'])

display(df_encoded[['Date', 'DayOfWeek_Monday', 'DayOfWeek_Tuesday']].head())

Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


,Date,DayOfWeek_Monday,DayOfWeek_Tuesday
0,2006-12-16,False,False
1,2006-12-16,False,False
2,2006-12-16,False,False
3,2006-12-16,False,False
4,2006-12-16,False,False
